# Stage 2 Notebook 71 - Exp2PPP Deeper ROI refinement (4 layers) + 72 sample points

**Architectural change #2: deeper per-prior feature aggregation.** The CLRKDLaneHead's ROI gather samples each prior's curve at `sample_points=36` positions, fuses across scales via scale_blocks, and runs `roi_refine_layers=3` iterative refinements. Each refinement re-samples the (refined) curve and updates again. NB62 used 36 sample points and 3 refinement stages.

Exp2PPP:
- `sample_points: 36 -> 72` -- double the points sampled along each prior's curve. The per-prior feature vector is now 2x longer before the FC projection, so each prior carries a denser feature signature.
- `roi_refine_layers: 3 -> 4` -- one extra iterative refinement stage. CLRNet's original code uses 3 stages but reports 4-5 can help on larger datasets. With 4 layers each prior gets one more chance to re-sample features along its refined curve.

Architectural diff vs NB62 (exp57):
- `model.lane_head.sample_points: 36 -> 72`
- `model.lane_head.roi_refine_layers: 3 -> 4`

Both increase the lane head's compute by ~50%. Wall-clock per epoch: ~1.6x NB62's.

Reference: CLRNet (Zheng et al. CVPR 2022) -- iterative ROI gather with curve refinement is the published mechanism. We are extending it slightly.

### Run mode
1. Smoke.
2. 12 epochs full 70K. ~3.5-4 hr (deeper head).

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp66_rmt_gca_anchor_cls_sep_vfl_deep_roi_full_data_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp66_rmt_gca_anchor_cls_sep_vfl_deep_roi_full_data_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp66_rmt_gca_anchor_cls_sep_vfl_deep_roi_full_data_joint_smoke.log
OK exp66_rmt_gca_anchor_cls_sep_vfl_deep_roi_full_data_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.2880 det_loss=3.4552 grad_cos=-0.3798 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.49535027146339417, 'gate/lane_mean': 0.5017317533493042, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp66_rmt_gca_anchor_cls_sep_vfl_deep_roi_full_data_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full12'
    EPOCHS = 12
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp66_rmt_gca_anchor_cls_sep_vfl_deep_roi_full_data_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp66_rmt_gca_anchor_cls_sep_vfl_deep_roi_full_data_joint_full12 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp66_rmt_gca_anchor_cls_sep_vfl_deep_roi_full_data_joint_full12.tar --epochs 12 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp66_rmt_gca_anchor_cls_sep_vfl_deep_roi_full_data_joint_full12.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp66_rmt_gca_anchor_cls_sep_vfl_deep_roi_full_data_joint_full12_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs

0

## What to watch in Exp2PPP

Reference NB62 (sample_points=36, refine_layers=3): matched_iou=0.550, decoded_f1=0.073, val_lane_f1=0.118.

Pass criteria at epoch 12:
- val/matched_line_iou >= 0.58 (deeper refinement gives more iterations of curve update).
- val/lane/decoded_f1 >= 0.08 (10 % over NB62).
- val/lane_f1 >= 0.13.
- pos-neg gap >= 0.05.
- val/lane/aux0, aux1, aux2 (intermediate stage losses) should DECREASE monotonically -- evidence that each refinement stage is producing better curves than the previous.

If matched_iou pushes >= 0.60: the geometry was capacity-limited at the ROI feature level, and the extra refinement stage unlocks it. Combine with high-res mask aux (Exp2OOO) for compounding gains.